<a href="https://colab.research.google.com/github/helenjoy/tiny-ml/blob/main/3_tinyml_quantization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**uploading raw Phyphox CSV data** to **training a Keras model, converting to 3 TFLite quantized formats, and benchmarking size, latency, and accuracy**.

---

 End-to-End TinyML Pipeline & Model Quantization Benchmarking



By completing this practical lab, students will be able to:

1. **Ingest Real Sensor Data:** Load raw time-series sensor data (`.csv`) recorded from a smartphone using **Phyphox**.
2. **Apply DSP Feature Extraction:** Implement sliding time windows and extract statistical features (**RMS**, **Standard Deviation**, **Peak-to-Peak**).
3. **Train an Edge Model:** Build and train a lightweight Multi-Layer Perceptron (MLP) classifier using TensorFlow/Keras.
4. **Apply Quantization Techniques:** Convert the trained floating-point model into **3 TFLite variants** (Unquantized Float32, Dynamic Range Int8, Full Int8).
5. **Evaluate TinyML Trade-offs:** Measure and compare model **Size (KB)**, **Inference Latency ($\mu$s)**, and **Classification Accuracy (%)**.

---

## 🛠️ Stage 1: Environment Setup & Interactive File Upload

### 📘 Concept Explanation

Microcontrollers process real-world signals recorded by hardware sensors. We start by uploading raw smartphone sensor data recorded with **Phyphox** (e.g., Accelerometer or Gyroscope logs).

Run the cell below to launch an upload widget in Google Colab:

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Set random seeds for reproducible results across classroom runs
np.random.seed(42)
tf.random.set_seed(42)

print("📂 Please select and upload your exported Phyphox CSV file...")
uploaded = files.upload()

# Dynamically extract the uploaded CSV filename
csv_file_path = list(uploaded.keys())[0]
print(f"✅ [Success] File uploaded: {csv_file_path}\n")

📂 Please select and upload your exported Phyphox CSV file...


Saving Raw Data.csv to Raw Data (1).csv
✅ [Success] File uploaded: Raw Data (1).csv



---

Stage 2: Data Ingestion & Sampling Rate Detection

### 📘 Concept Explanation

Phyphox records time-series values alongside a `Time (s)` timestamp column. To process sensor streams correctly, we must determine the **sampling frequency ($f_s$)** in Hertz ($\text{Hz}$):

$$f_s = \frac{1}{\Delta t_{\text{mean}}} = \frac{1}{\text{Mean}(t_{i+1} - t_i)}$$

In [ ]:
def load_phyphox_data(csv_path):
    """
    Parses exported Phyphox CSV files and extracts time and 3-axis motion columns.
    Handles Accelerometer (m/s^2) or Gyroscope (rad/s) exported columns automatically.
    """
    df = pd.read_csv(csv_path)
    time_col = df['Time (s)'].values

    # Auto-detect sensor column header formats
    if 'Acceleration x (m/s^2)' in df.columns:
        x = df['Acceleration x (m/s^2)'].values
        y = df['Acceleration y (m/s^2)'].values
        z = df['Acceleration z (m/s^2)'].values
    elif 'Gyroscope x (rad/s)' in df.columns:
        x = df['Gyroscope x (rad/s)'].values
        y = df['Gyroscope y (rad/s)'].values
        z = df['Gyroscope z (rad/s)'].values
    else:
        # Fallback for generic 3-axis CSV formats
        x, y, z = df.iloc[:, 1].values, df.iloc[:, 2].values, df.iloc[:, 3].values

    return time_col, x, y, z

# Load raw signal
time_s, x_raw, y_raw, z_raw = load_phyphox_data(csv_file_path)

# Dynamically measure sensor sampling frequency
sample_rate = int(round(1.0 / np.mean(np.diff(time_s))))
print(f"📊 Signal loaded successfully!")
print(f"⏱️ Measured Sampling Rate: {sample_rate} Hz")

📊 Signal loaded successfully!
⏱️ Measured Sampling Rate: 100 Hz


 ### Stage 3: Sliding Window Preprocessing & Feature Extraction (DSP)

### 📘 Concept Explanation

Microcontrollers cannot pass hundreds of raw sensor readings into a model every second due to RAM limits. Instead, we divide the data stream into **sliding time windows** and compute low-dimensional summary features:

1. **Combined Vector Magnitude:** Removes orientation dependency ($x, y, z$ axes direction):

$$\text{mag} = \sqrt{x^2 + y^2 + z^2}$$


2. **Root Mean Square (RMS):** Quantifies overall motion energy:

$$\text{RMS} = \sqrt{\frac{1}{N} \sum_{i=1}^{N} \text{mag}_i^2}$$


3. **Standard Deviation ($\sigma$):** Measures turbulence and vibration intensity (Key feature for separating **Idle** vs **Motion**):

$$\sigma = \sqrt{\frac{1}{N} \sum_{i=1}^{N} (\text{mag}_i - \mu)^2}$$


4. **Peak-to-Peak ($V_{pp}$):** Measures dynamic range ($\text{Max} - \text{Min}$).

In [ ]:
def preprocess_and_extract_features(x, y, z, sample_rate=50, window_sec=2.0, step_sec=0.2):
    """
    Applies sliding window segmentation (2.0s duration, 0.2s stride)
    and calculates statistical features per frame.
    """
    window_size = int(sample_rate * window_sec)
    step_size = int(sample_rate * step_sec)

    mag = np.sqrt(x**2 + y**2 + z**2)

    feature_matrix = []
    labels = []

    for start in range(0, len(mag) - window_size + 1, step_size):
        end = start + window_size
        win_mag = mag[start:end]
        win_x, win_y, win_z = x[start:end], y[start:end], z[start:end]

        # Calculate time-domain DSP features
        rms_mag = np.sqrt(np.mean(win_mag**2))  # Energy
        std_mag = np.std(win_mag)                # Vibration / Turbulence
        p2p_mag = np.ptp(win_mag)                # Dynamic Range

        rms_x = np.sqrt(np.mean(win_x**2))
        rms_y = np.sqrt(np.mean(win_y**2))
        rms_z = np.sqrt(np.mean(win_z**2))

        feature_vector = [rms_mag, std_mag, p2p_mag, rms_x, rms_y, rms_z]
        feature_matrix.append(feature_vector)

        # Ground-truth binary labeling: Low variance = Idle (0), High variance = Motion (1)
        label = 1 if std_mag > 0.4 else 0
        labels.append(label)

    return np.array(feature_matrix, dtype=np.float32), np.array(labels, dtype=np.int32)

# Run Feature Extraction
X_data, y_data = preprocess_and_extract_features(x_raw, y_raw, z_raw, sample_rate=sample_rate)

print(f"🔢 Matrix Shape: Extracted {X_data.shape[0]} windows with {X_data.shape[1]} features each.")
print(f"🏷️ Class Distribution -> Class 0 (Idle): {np.sum(y_data == 0)}, Class 1 (Motion): {np.sum(y_data == 1)}\n")

# Split dataset into Training (80%) and Test (20%) sets
split = int(0.8 * len(X_data))
X_train, X_test = X_data[:split], X_data[split:]
y_train, y_test = y_data[:split], y_data[split:]

🔢 Matrix Shape: Extracted 202 windows with 6 features each.
🏷️ Class Distribution -> Class 0 (Idle): 127, Class 1 (Motion): 75



---

## Stage 4: Baseline Keras Model Training

### 📘 Concept Explanation

We construct a compact Multi-Layer Perceptron (MLP) classifier suited for microcontrollers.

In [ ]:
num_features = X_data.shape[1]
num_classes = 2

# Define a compact Edge-friendly neural network
model = keras.Sequential([
    layers.Input(shape=(num_features,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(8, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Train Model
print("🏋️ Training Baseline Keras Model...")
model.fit(X_train, y_train, epochs=30, batch_size=16, verbose=0)

float_acc = model.evaluate(X_test, y_test, verbose=0)[1]
print(f"🎯 [Success] Baseline Keras Model Test Accuracy: {float_acc * 100:.2f}%\n")

🏋️ Training Baseline Keras Model...
🎯 [Success] Baseline Keras Model Test Accuracy: 92.68%



---

## 🗜️ Stage 5: TFLite Quantization Engine

### 📘 Concept Explanation for Students

Quantization transforms 32-bit floating-point (`float32`) parameters into 8-bit integers (`int8`).

```
Float32 (4 Bytes) ➔ [Quantization Transformation] ➔ Int8 (1 Byte)  ==>  4x Memory Reduction!

```

### Quantization Variants



* **Unquantized (Float32 Baseline):** Standard model format with high precision, but requiring large storage memory and floating-point math operations.
* **Post-Training Dynamic Range Quantization (Int8 Weights, Float32 Activations):** Quantizes weights to 8-bit integers at conversion time while keeping activations in floating point during execution.
* **Full Integer Post-Training Quantization (Full Int8):** Quantizes both weights and activations to 8-bit integers using a `representative_dataset` generator during conversion. This enables deployment on ultra-low-power microcontrollers lacking Floating-Point Units (FPUs).
* **Quantization-Aware Training (QAT):** Simulates low-precision quantization errors during model training so the neural network learns to compensate for rounding errors before deployment.

In [ ]:
print("⚡ Converting Model into 3 TFLite Quantized Formats...")

# Variant A: Unquantized Float32 TFLite Model
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_float_model = converter.convert()
with open('model_float.tflite', 'wb') as f:
    f.write(tflite_float_model)

# Variant B: Dynamic Range Int8 Quantization
converter_dynamic = tf.lite.TFLiteConverter.from_keras_model(model)
converter_dynamic.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_dynamic_model = converter_dynamic.convert()
with open('model_dynamic_int8.tflite', 'wb') as f:
    f.write(tflite_dynamic_model)

# Variant C: Full Integer Int8 Quantization (with Representative Dataset)
def representative_dataset_generator():
    """Generates sample training inputs to calibrate int8 dynamic ranges for activations."""
    for sample in X_train[:100]:
        yield [np.expand_dims(sample, axis=0)]

converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_dataset_generator

# The following line is causing an AttributeError. When `inference_input_type` and
# `inference_output_type` are set to tf.int8, the converter will implicitly
# target integer operations for full quantization.
# converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTIN]

# Enforce 8-bit integer inputs and outputs for full quantization
converter_int8.inference_input_type = tf.int8
converter_int8.inference_output_type = tf.int8

tflite_int8_model = converter_int8.convert()
with open('model_full_int8.tflite', 'wb') as f:
    f.write(tflite_int8_model)

print("✅ [Success] All 3 TFLite models generated and saved to disk.\n")

⚡ Converting Model into 3 TFLite Quantized Formats...
Saved artifact at '/tmp/tmpdmsj5k9_'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 6), dtype=tf.float32, name='keras_tensor_4')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  132688003067024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132688003064336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132688003070672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132688003068560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132688003067792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132688003070864: TensorSpec(shape=(), dtype=tf.resource, name=None)
Saved artifact at '/tmp/tmpocmy7v86'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 6), dtype=tf.float32, name='keras_tensor_4')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.floa

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


✅ [Success] All 3 TFLite models generated and saved to disk.



---

##  Stage 6: Benchmarking Framework (Memory, Latency, Accuracy)

### 📘 Concept Explanation

We evaluate three core TinyML metrics across each model variant:

* **Memory Footprint (KB):** Flash/Storage space required to store `.tflite` model files.
* **Execution Latency ($\mu$s):** Microseconds elapsed per individual model inference invocation.
* **Accuracy (%):** Classification performance on unseen test data after quantization rounding.

In [ ]:
def evaluate_tflite_model(model_path, X_eval, y_eval):
    """
    Measures file size (KB), average execution latency (microseconds),
    and classification accuracy (%) for a TFLite model file.
    """
    # 1. Measure Flash Memory footprint
    file_size_kb = os.path.getsize(model_path) / 1024.0

    # 2. Initialize TFLite Interpreter
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    input_dtype = input_details[0]['dtype']
    output_dtype = output_details[0]['dtype']

    correct_preds = 0
    latencies = []

    # 3. Benchmark Sample-by-Sample
    for i in range(len(X_eval)):
        input_data = np.expand_dims(X_eval[i], axis=0).astype(np.float32)

        # Quantize inputs dynamically if the model expects INT8 / UINT8
        if input_dtype == np.int8 or input_dtype == np.uint8:
            scale, zero_point = input_details[0]['quantization']
            if scale != 0:
                input_data = (input_data / scale + zero_point)
            input_data = np.round(input_data).astype(input_dtype)

        # Measure precise inference execution speed
        start_time = time.perf_counter()
        interpreter.set_tensor(input_details[0]['index'], input_data)
        interpreter.invoke()
        output_data = interpreter.get_tensor(output_details[0]['index'])
        end_time = time.perf_counter()

        latencies.append((end_time - start_time) * 1e6)  # Microseconds (µs)

        # De-quantize outputs dynamically if the model returned INT8 / UINT8
        if output_dtype == np.int8 or output_dtype == np.uint8:
            scale_out, zero_point_out = output_details[0]['quantization']
            output_data = (output_data.astype(np.float32) - zero_point_out) * scale_out

        pred_class = np.argmax(output_data)
        if pred_class == y_eval[i]:
            correct_preds += 1

    accuracy = (correct_preds / len(X_eval)) * 100.0 if len(X_eval) > 0 else 0.0
    avg_latency_us = np.mean(latencies) if len(latencies) > 0 else 0.0

    return file_size_kb, avg_latency_us, accuracy

# Run Benchmark Evaluation across all 3 variants
float_size, float_lat, float_acc = evaluate_tflite_model('model_float.tflite', X_test, y_test)
dyn_size, dyn_lat, dyn_acc = evaluate_tflite_model('model_dynamic_int8.tflite', X_test, y_test)
int8_size, int8_lat, int8_acc = evaluate_tflite_model('model_full_int8.tflite', X_test, y_test)

# Print Detailed Benchmark Results Table
print("\n" + "=" * 65)
print(f"{'Model Variant':<22} | {'Size (KB)':<10} | {'Latency (μs)':<12} | {'Accuracy (%)':<10}")
print("=" * 65)
print(f"{'Float32 (Unquantized)':<22} | {float_size:<10.2f} | {float_lat:<12.2f} | {float_acc:<10.2f}")
print(f"{'Dynamic Range Int8':<22} | {dyn_size:<10.2f} | {dyn_lat:<12.2f} | {dyn_acc:<10.2f}")
print(f"{'Full Integer Int8':<22} | {int8_size:<10.2f} | {int8_lat:<12.2f} | {int8_acc:<10.2f}")
print("=" * 65)


Model Variant          | Size (KB)  | Latency (μs) | Accuracy (%)
Float32 (Unquantized)  | 3.12       | 14.21        | 92.68     
Dynamic Range Int8     | 3.12       | 13.19        | 92.68     
Full Integer Int8      | 3.31       | 16.44        | 92.68     


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)




1. **Memory Storage:** Why does **Full Int8** quantization reduce model size compared to **Float32**?
2. **Hardware Constraints:** Which quantization variant is required for deployment on microcontrollers without hardware Floating-Point Units (FPUs)?
3. **Trade-offs:** Did converting the model from Float32 to Full Int8 significantly impact classification accuracy? Why or why not?